In [1]:
import pandas as pd

In [2]:
cards_df = pd.read_csv('data/mtgjson.com/cards.csv')
cards_df.head()

C:\Users\sophc\AppData\Local\Temp\ipykernel_30352\3730294610.py:1: DtypeWarning: Columns (2,3,7,12,16,19,21,24,26,27,32,34,35,37,39,41,42,43,49,54,55,59,60,61,65,66,70,72) have mixed types. Specify dtype option on import or set low_memory=False.
  cards_df = pd.read_csv('data/mtgjson.com/cards.csv')


,artist,artistIds,asciiName,attractionLights,availability,boosterTypes,borderColor,cardParts,colorIdentity,colorIndicator,...,subsets,subtypes,supertypes,text,toughness,type,types,uuid,variations,watermark
0,Pete Venters,d54c4a1a-c0c5-4834-84db-125d341f3ad8,NaN,NaN,"mtgo, paper",default,black,NaN,W,NaN,...,NaN,"Human, Cleric",NaN,First strike (This creature deals combat damag...,4,Creature — Human Cleric,Creature,5f8287b1-5bb6-5f4c-ad17-316a40d5bb0c,b7c19924-b4bf-56fc-aa73-f586e940bd42,NaN
1,Pete Venters,d54c4a1a-c0c5-4834-84db-125d341f3ad8,NaN,NaN,"mtgo, paper",default,black,NaN,W,NaN,...,NaN,"Human, Cleric",NaN,First strike (This creature deals combat damag...,4,Creature — Human Cleric,Creature,b7c19924-b4bf-56fc-aa73-f586e940bd42,5f8287b1-5bb6-5f4c-ad17-316a40d5bb0c,NaN
2,Volkan Baǵa,93bec3c0-0260-4d31-8064-5d01efb4153f,NaN,NaN,"mtgo, paper",default,black,NaN,W,NaN,...,NaN,Angel,NaN,"Flying\nWhen this creature enters, you gain 3 ...",3,Creature — Angel,Creature,57aaebc1-850c-503d-9f6e-bb8d00d8bf7c,8fd4e2eb-3eb4-50ea-856b-ef638fa47f8a,NaN
3,Volkan Baǵa,93bec3c0-0260-4d31-8064-5d01efb4153f,NaN,NaN,"mtgo, paper",default,black,NaN,W,NaN,...,NaN,Angel,NaN,"Flying\nWhen this creature enters, you gain 3 ...",3,Creature — Angel,Creature,8fd4e2eb-3eb4-50ea-856b-ef638fa47f8a,57aaebc1-850c-503d-9f6e-bb8d00d8bf7c,NaN
4,Mark Zug,48e2b98c-5467-4671-bd42-4c3746115117,NaN,NaN,"mtgo, paper",default,black,NaN,W,NaN,...,NaN,NaN,NaN,Target creature gets +3/+3 and gains flying un...,NaN,Sorcery,Sorcery,55bd38ca-dc73-5c06-8f80-a6ddd2f44382,c5655330-5131-5f40-9d3e-0549d88c6e9e,NaN


In [3]:
cards_df.columns

Index(['artist', 'artistIds', 'asciiName', 'attractionLights', 'availability',
       'boosterTypes', 'borderColor', 'cardParts', 'colorIdentity',
       'colorIndicator', 'colors', 'defense', 'duelDeck', 'edhrecRank',
       'edhrecSaltiness', 'faceConvertedManaCost', 'faceFlavorName',
       'faceManaValue', 'faceName', 'facePrintedName', 'finishes',
       'flavorName', 'flavorText', 'frameEffects', 'frameVersion', 'hand',
       'hasAlternativeDeckLimit', 'hasContentWarning', 'hasFoil', 'hasNonFoil',
       'isAlternative', 'isFullArt', 'isFunny', 'isGameChanger',
       'isOnlineOnly', 'isOversized', 'isPromo', 'isRebalanced', 'isReprint',
       'isReserved', 'isStarter', 'isStorySpotlight', 'isTextless',
       'isTimeshifted', 'keywords', 'language', 'layout', 'leadershipSkills',
       'life', 'loyalty', 'manaCost', 'manaValue', 'name', 'number',
       'originalPrintings', 'originalReleaseDate', 'originalText',
       'otherFaceIds', 'power', 'printedName', 'printedText', 'pr

In [4]:
# some of these columns only have True values, with 
# values that would otherwise be false being NaN.

# 'hasAlternativeDeckLimit', 'hasContentWarning' - too little count
# 'isFullArt', 'isFunny', 'isOversized' - unnecessary
# 'isStarter' - unnecessary AND deprecated
# 'isRebalanced', 'isStorySpotlight', 'isTextless', 'isTimeshifted' - not what we're looking for?

cards_df[['isAlternative', 'isGameChanger', 'isOnlineOnly', 
          'isPromo', 'isReprint', 'isReserved']] = cards_df[['isAlternative', 'isGameChanger', 'isOnlineOnly', 
                                                             'isPromo', 'isReprint', 'isReserved']].fillna(False)

cards_df = cards_df[cards_df['isOnlineOnly'] == False]

In [5]:
# filter to paper and mtgo because price_df only lists those two
cards_df = cards_df[(cards_df['availability'].str.contains('paper')) | (cards_df['availability'].str.contains('mtgo'))]
cards_df['availability'].value_counts()

mtgo, paper           40854
paper                 38162
arena, mtgo, paper    17978
arena, paper           1158
Name: availability, dtype: int64

In [6]:
cards_df_eng = cards_df[cards_df['language'] == 'English']

# don't need artists?
# do we need borderColor? flavorText?

cards_df_feats = cards_df_eng[[
                            'uuid', 'name', 'availability',
                            'colorIdentity', 'defense', 
                            'edhrecRank', 'edhrecSaltiness', 
                            'finishes', 'isAlternative', 
                            'isGameChanger', 'isPromo', 
                            'isReprint', 'isReserved', 
                            'keywords', 'layout', 
                            'loyalty', 'manaCost','manaValue', 
                            'number', 'otherFaceIds', 'power', 
                            'rarity', 'setCode', 'subtypes', 
                            'supertypes', 'text', 'toughness', 
                            'type', 'types', 'variations'
                            ]]

cards_df_feats = cards_df_feats.rename(columns={'name': 'cardName', 'number': 'cardNumber'})

cards_df_feats.head()

,uuid,cardName,availability,colorIdentity,defense,edhrecRank,edhrecSaltiness,finishes,isAlternative,isGameChanger,...,power,rarity,setCode,subtypes,supertypes,text,toughness,type,types,variations
0,5f8287b1-5bb6-5f4c-ad17-316a40d5bb0c,Ancestor's Chosen,"mtgo, paper",W,NaN,23684.0,0.27,nonfoil,False,False,...,4,uncommon,10E,"Human, Cleric",NaN,First strike (This creature deals combat damag...,4,Creature — Human Cleric,Creature,b7c19924-b4bf-56fc-aa73-f586e940bd42
1,b7c19924-b4bf-56fc-aa73-f586e940bd42,Ancestor's Chosen,"mtgo, paper",W,NaN,23684.0,0.27,foil,False,False,...,4,uncommon,10E,"Human, Cleric",NaN,First strike (This creature deals combat damag...,4,Creature — Human Cleric,Creature,5f8287b1-5bb6-5f4c-ad17-316a40d5bb0c
2,57aaebc1-850c-503d-9f6e-bb8d00d8bf7c,Angel of Mercy,"mtgo, paper",W,NaN,18663.0,NaN,nonfoil,False,False,...,3,uncommon,10E,Angel,NaN,"Flying\nWhen this creature enters, you gain 3 ...",3,Creature — Angel,Creature,8fd4e2eb-3eb4-50ea-856b-ef638fa47f8a
3,8fd4e2eb-3eb4-50ea-856b-ef638fa47f8a,Angel of Mercy,"mtgo, paper",W,NaN,18663.0,NaN,foil,False,False,...,3,uncommon,10E,Angel,NaN,"Flying\nWhen this creature enters, you gain 3 ...",3,Creature — Angel,Creature,57aaebc1-850c-503d-9f6e-bb8d00d8bf7c
4,55bd38ca-dc73-5c06-8f80-a6ddd2f44382,Angelic Blessing,"mtgo, paper",W,NaN,25093.0,0.19,nonfoil,False,False,...,NaN,common,10E,NaN,NaN,Target creature gets +3/+3 and gains flying un...,NaN,Sorcery,Sorcery,c5655330-5131-5f40-9d3e-0549d88c6e9e


In [7]:
prices_df = pd.read_csv('data/mtgjson.com/cardPrices.csv')

prices_df = prices_df[prices_df['currency'] == 'USD']
# dates gathered 2026-01-03

print(prices_df['priceProvider'].value_counts())

prices_df.head()

cardkingdom    208497
manapool       142758
tcgplayer      142424
cardhoarder     79777
cardsphere      76114
Name: priceProvider, dtype: int64


,cardFinish,currency,date,gameAvailability,price,priceProvider,providerListing,uuid
0,normal,USD,2026-01-03,mtgo,0.18,cardhoarder,retail,f182e364-0439-5594-a6e6-75f7889ccf45
1,normal,USD,2026-01-03,mtgo,0.33,cardhoarder,retail,330deaa3-dd7a-52a8-bfbc-b323cd16a409
2,normal,USD,2026-01-03,mtgo,0.02,cardhoarder,retail,79e36956-b91f-580f-8309-7d9585a67560
3,normal,USD,2026-01-03,mtgo,0.33,cardhoarder,retail,6afb2b4c-530a-57d5-8e7f-871239f6fa05
4,normal,USD,2026-01-03,mtgo,0.02,cardhoarder,retail,b1fc2762-92aa-5a14-8509-a59cb611e376


In [8]:
average_prices = prices_df.groupby(['uuid'])['price'].mean().reset_index()
average_prices.head()

,uuid,price
0,00010d56-fe38-5e35-8aed-518019aa36a5,7.910000
1,0001e0d0-2dcd-5640-aadc-a84765cf5fc9,4.715000
2,0003caab-9ff5-5d1a-bc06-976dd0457f19,0.539000
3,0003d249-25d9-5223-af1e-1130f09622a7,0.234444
4,0004822c-c181-5564-808d-a6cc48359a1a,1.063333


In [9]:
legal_df = pd.read_csv('data/mtgjson.com/cardLegalities.csv')

legal_df.head()

C:\Users\sophc\AppData\Local\Temp\ipykernel_30352\2955574431.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  legal_df = pd.read_csv('data/mtgjson.com/cardLegalities.csv')


,alchemy,brawl,commander,duel,future,gladiator,historic,legacy,modern,oathbreaker,...,paupercommander,penny,pioneer,predh,premodern,standard,standardbrawl,timeless,uuid,vintage
0,NaN,NaN,Legal,Legal,NaN,NaN,NaN,Legal,Legal,Legal,...,NaN,Legal,NaN,Legal,Legal,NaN,NaN,NaN,5f8287b1-5bb6-5f4c-ad17-316a40d5bb0c,Legal
1,NaN,NaN,Legal,Legal,NaN,NaN,NaN,Legal,Legal,Legal,...,NaN,Legal,NaN,Legal,Legal,NaN,NaN,NaN,b7c19924-b4bf-56fc-aa73-f586e940bd42,Legal
2,NaN,Legal,Legal,Legal,NaN,Legal,Legal,Legal,Legal,Legal,...,Legal,Legal,NaN,Legal,Legal,NaN,NaN,Legal,57aaebc1-850c-503d-9f6e-bb8d00d8bf7c,Legal
3,NaN,Legal,Legal,Legal,NaN,Legal,Legal,Legal,Legal,Legal,...,Legal,Legal,NaN,Legal,Legal,NaN,NaN,Legal,8fd4e2eb-3eb4-50ea-856b-ef638fa47f8a,Legal
4,NaN,NaN,Legal,Legal,NaN,NaN,NaN,Legal,Legal,Legal,...,Legal,Legal,NaN,Legal,Legal,NaN,NaN,NaN,55bd38ca-dc73-5c06-8f80-a6ddd2f44382,Legal


In [10]:
# keep it to formats that have many legal cards
# but also seem semi-popular, needs review

print(legal_df['commander'].value_counts())
print(legal_df['legacy'].value_counts())
print(legal_df['standard'].value_counts())

legal_df_most_cards = legal_df[['commander', 'legacy', 'standard', 'vintage', 'uuid']]

Legal     101057
Banned       350
Name: commander, dtype: int64
Legal     100375
Banned       897
Name: legacy, dtype: int64
Legal     16144
Banned       28
Name: standard, dtype: int64


In [11]:
sets_df = pd.read_csv('data/mtgjson.com/sets.csv')

# needed for release date

sets_df.head()

,baseSetSize,block,cardsphereSetId,code,isFoilOnly,isForeignOnly,isNonFoilOnly,isOnlineOnly,isPartialPreview,keyruneCode,...,mcmIdExtras,mcmName,mtgoCode,name,parentCode,releaseDate,tcgplayerGroupId,tokenSetCode,totalSetSize,type
0,383,Core Set,755.0,10E,False,NaN,NaN,False,NaN,10E,...,NaN,Tenth Edition,10E,Tenth Edition,NaN,2007-07-13,1.0,T10E,510,core
1,302,Core Set,938.0,2ED,False,NaN,True,False,NaN,2ED,...,NaN,NaN,NaN,Unlimited Edition,NaN,1993-12-01,115.0,NaN,302,core
2,331,NaN,1462.0,2X2,False,NaN,NaN,False,NaN,2X2,...,5071.0,Double Masters 2022,NaN,Double Masters 2022,NaN,2022-07-08,3070.0,T2X2,579,masters
3,332,NaN,1251.0,2XM,False,NaN,NaN,False,NaN,2XM,...,3209.0,Double Masters,2XM,Double Masters,NaN,2020-08-07,2655.0,T2XM,384,masters
4,594,NaN,NaN,30A,False,NaN,True,False,NaN,30A,...,NaN,30th Anniversary Edition,NaN,30th Anniversary Edition,NaN,2022-11-28,3178.0,T30A,594,memorabilia


In [12]:
sets_df_real_eng = sets_df[(sets_df['isPartialPreview'] != True) & 
                        (sets_df['isForeignOnly'] != True) & 
                        (sets_df['isOnlineOnly'] != True)]

# "type": "expType"
# don't need type?

sets_df_real_eng = sets_df_real_eng[['code', 'name', 'releaseDate']]
sets_df_real_eng = sets_df_real_eng.rename(columns={"code": "setCode", "name": "setName"})
sets_df_real_eng.head()

,setCode,setName,releaseDate
0,10E,Tenth Edition,2007-07-13
1,2ED,Unlimited Edition,1993-12-01
2,2X2,Double Masters 2022,2022-07-08
3,2XM,Double Masters,2020-08-07
4,30A,30th Anniversary Edition,2022-11-28


In [13]:
card_ids_df = pd.read_csv("data/mtgjson.com/cardIdentifiers.csv")

# 'cardKingdomId', 'cardsphereId' are priceProviders in price_df
# but price_df will be averaged, so not needed?

card_ids_df = card_ids_df[['uuid', 'scryfallId', 'tcgplayerProductId', 'multiverseId']]

card_ids_df.head()

C:\Users\sophc\AppData\Local\Temp\ipykernel_30352\373716369.py:1: DtypeWarning: Columns (9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  card_ids_df = pd.read_csv("data/mtgjson.com/cardIdentifiers.csv")


,uuid,scryfallId,tcgplayerProductId,multiverseId
0,5f8287b1-5bb6-5f4c-ad17-316a40d5bb0c,7a5cd03c-4227-4551-aa4b-7d119f0468b5,15032.0,130550.0
1,b7c19924-b4bf-56fc-aa73-f586e940bd42,82072a1d-c1ab-4b4f-875f-d0591447e0a4,15032.0,NaN
2,57aaebc1-850c-503d-9f6e-bb8d00d8bf7c,8f7980d4-da43-4d6d-ad16-14b8a34ae91d,15033.0,129465.0
3,8fd4e2eb-3eb4-50ea-856b-ef638fa47f8a,b0157252-6949-4f03-a15c-c168512123a8,15033.0,NaN
4,55bd38ca-dc73-5c06-8f80-a6ddd2f44382,a285aa3f-bcfb-4fc3-8441-85a56c72a3e4,15035.0,129711.0


In [14]:
print(len(cards_df_feats), len(average_prices))

card_prices_df = cards_df_feats.merge(average_prices, how='right', on='uuid')
print(len(card_prices_df))

95685 104075
104075


In [15]:
card_prices_legal_df = card_prices_df.merge(legal_df_most_cards, on='uuid')
print(len(card_prices_legal_df))

99736


In [16]:
card_prices_sets_df = card_prices_legal_df.merge(sets_df_real_eng, how='left', on='setCode')
print(len(card_prices_sets_df))

99736


In [17]:
card_prices_sets_ids_df = card_prices_sets_df.merge(card_ids_df, on='uuid')
print(len(card_prices_sets_ids_df))

99736


In [18]:
card_prices_sets_legal_df = card_prices_sets_ids_df[(card_prices_sets_ids_df['commander'] == 'Legal') |
                                                (card_prices_sets_ids_df['legacy'] == 'Legal') |
                                                (card_prices_sets_ids_df['standard'] == 'Legal')]

# I don't think having unreleased cards is good
card_prices_sets_legal_df = card_prices_sets_legal_df[card_prices_sets_legal_df['releaseDate'].notnull()]

print(len(card_prices_sets_legal_df))
card_prices_sets_legal_df.head()


89570


,uuid,cardName,availability,colorIdentity,defense,edhrecRank,edhrecSaltiness,finishes,isAlternative,isGameChanger,...,price,commander,legacy,standard,vintage,setName,releaseDate,scryfallId,tcgplayerProductId,multiverseId
0,00010d56-fe38-5e35-8aed-518019aa36a5,Sphinx of the Final Word,paper,U,NaN,10186.0,0.14,foil,False,False,...,7.910000,Legal,Legal,Legal,Legal,Oath of the Gatewatch Promos,2016-01-22,f6555d1f-d4cf-41f7-99d3-88fd53e75457,111268.0,NaN
1,0001e0d0-2dcd-5640-aadc-a84765cf5fc9,Goblin King,paper,R,NaN,3454.0,0.34,nonfoil,False,False,...,4.715000,Legal,Legal,NaN,Legal,Revised Edition,1994-04-11,e3094187-d666-414b-a1fd-ae0ef55c3fcb,1435.0,1296.0
2,0003caab-9ff5-5d1a-bc06-976dd0457f19,Caravan Vigil,"mtgo, paper",G,NaN,12489.0,0.08,"nonfoil, foil",False,False,...,0.539000,Legal,Legal,NaN,Legal,Innistrad,2011-09-30,9a8dfb98-a975-41bf-8aac-c0001c9ddaa7,56309.0,234444.0
3,0003d249-25d9-5223-af1e-1130f09622a7,Deadshot Minotaur,"mtgo, paper","G, R",NaN,25150.0,0.20,"nonfoil, foil",False,False,...,0.234444,Legal,Legal,NaN,Legal,Alara Reborn,2009-04-30,aacb131b-74c9-4e6c-9466-27710bc9441f,31714.0,179543.0
4,0004822c-c181-5564-808d-a6cc48359a1a,Clone Legion,"arena, mtgo, paper",U,NaN,3488.0,0.48,"nonfoil, foil",False,False,...,1.063333,Legal,Legal,NaN,Legal,Avatar: The Last Airbender Eternal,2025-11-21,b7ea0b25-19e3-4d75-9daa-9110dce7e6fd,661987.0,NaN


In [19]:
card_prices_sets_legal_df.columns

Index(['uuid', 'cardName', 'availability', 'colorIdentity', 'defense',
       'edhrecRank', 'edhrecSaltiness', 'finishes', 'isAlternative',
       'isGameChanger', 'isPromo', 'isReprint', 'isReserved', 'keywords',
       'layout', 'loyalty', 'manaCost', 'manaValue', 'cardNumber',
       'otherFaceIds', 'power', 'rarity', 'setCode', 'subtypes', 'supertypes',
       'text', 'toughness', 'type', 'types', 'variations', 'price',
       'commander', 'legacy', 'standard', 'vintage', 'setName', 'releaseDate',
       'scryfallId', 'tcgplayerProductId', 'multiverseId'],
      dtype='object')

In [20]:
print(card_prices_sets_legal_df['isReprint'].value_counts())
print('---')
print(card_prices_sets_legal_df['availability'].value_counts())
print('---')
print(card_prices_sets_legal_df['layout'].value_counts())

True     50189
False    39381
Name: isReprint, dtype: int64
---
mtgo, paper           40629
paper                 30053
arena, mtgo, paper    17731
arena, paper           1157
Name: availability, dtype: int64
---
normal             85057
transform           1926
adventure            670
modal_dfc            478
split                418
saga                 343
reversible_card      150
aftermath            140
mutate               101
flip                  62
class                 61
leveler               60
meld                  42
prototype             38
case                  24
Name: layout, dtype: int64


In [21]:
print(card_prices_sets_legal_df['rarity'].value_counts())
print('---')
print(card_prices_sets_legal_df['subtypes'].value_counts())
print('---')
print(card_prices_sets_legal_df['supertypes'].value_counts())
print('---')
print(card_prices_sets_legal_df['types'].value_counts())

rare        33851
common      25073
uncommon    21791
mythic       8497
special       358
Name: rarity, dtype: int64
---
Aura                       2666
Equipment                  1490
Human, Wizard              1441
Human, Soldier             1223
Elemental                   979
                           ... 
Ogre, Samurai, Shaman         1
Human, Clown, Berserker       1
Gnome, Ranger                 1
Hippo, Ox                     1
Mole, Scout                   1
Name: subtypes, Length: 2523, dtype: int64
---
Legendary          12676
Basic               3270
Snow                 175
Basic, Snow           55
World                 37
Legendary, Snow       30
Name: supertypes, dtype: int64
---
Creature                  39909
Land                       9964
Instant                    9578
Sorcery                    9551
Enchantment                8318
Artifact                   7031
Artifact, Creature         2810
Planeswalker               1295
Enchantment, Creature       708
Artifac

In [22]:
card_prices_sets_legal_df.isna().sum()

uuid                      0
cardName                  0
availability              0
colorIdentity          9151
defense               89517
edhrecRank             3457
edhrecSaltiness       10240
finishes                  0
isAlternative             0
isGameChanger             0
isPromo                   0
isReprint                 0
isReserved                0
keywords              50535
layout                    0
loyalty               88281
manaCost              11200
manaValue                 0
cardNumber                0
otherFaceIds          85690
power                 45589
rarity                    0
setCode                   0
subtypes              33921
supertypes            73327
text                    904
toughness             45589
type                      0
types                     0
variations            58935
price                     0
commander                 0
legacy                  135
standard              74896
vintage                 135
setName             